# AGENTS4DQ — Agents for Data Quality

**REPLY × LUISS Machine Learning Project | A.Y. 2025/26**

| Member | Student ID |
|---|---|
| Maria Dichio | 822231 |
| Gianfranco Votta *(Captain)* | 807861 |
| Armando Fornario | 813811 |

---

## Overview

This notebook contains all the code used in the AGENTS4DQ project. The system is a multi-agent pipeline that automatically assesses and improves the quality of a raw CSV dataset through five sequential stages:

**Schema Validation → Completeness Analysis → Consistency Validation → Anomaly Detection → Remediation**

Each agent combines LLM-based reasoning (for column selection, placeholder detection, cross-column logic inference) with deterministic statistical checks (3σ outlier detection, frequency thresholds, exact duplicate counting). The LLM-driven approach makes the system **dataset-agnostic**: no hard-coded rules need to change when switching datasets.

We validate the pipeline on two structurally different NoiPA datasets:
1. `spesa.csv`: expenditure records (7,543 rows, 18 columns)
2. `attivazioniCessazioni.csv`: personnel activation/termination records (20,102 rows, 19 columns)

The notebook is structured as follows:
- **Section 0**: Environment setup and imports
- **Section 1**: Utility classes and functions
- **Section 2**: Agent class definitions
- **Section 3**: Experiment 1 - Full pipeline on `spesa.csv`
- **Section 4**: Experiment 2 - Full pipeline on `attivazioniCessazioni.csv`
- **Section 5**: Conclusions

## 0: Environment Setup & Imports

We begin by importing all the required libraries. The project relies on:
- **pandas / numpy** for data manipulation
- **langchain** integrations for LLM communication (Google GenAI for Gemma and Gemini models)
- **SQLite caching** to avoid redundant API calls during repeated runs: once an LLM response has been generated for a given prompt, it is stored locally and reused
- **.env** to load API keys from a `.env` file

A `.env` file must be present at the project root with the following keys:
```
GOOGLE_API_KEY=<your_google_genai_key>
```

In [1]:
import sys
import os
import re
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

CACHE_DB_PATH = "notebook_cache.db"
set_llm_cache(SQLiteCache(database_path=CACHE_DB_PATH))

print("Environment ready.")
print(f"Python {sys.version}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

Environment ready.
Python 3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]
pandas 2.2.3, numpy 2.4.3


## 1: Utility Classes and Functions

Before defining the agents, we need three foundational utilities:

1. **`Outputs`**: a wrapper class for LLM responses. Different models return content in different formats (plain strings, lists with thinking/text blocks, markdown-fenced JSON). This class normalises all of them into clean text or parsed JSON.

2. **`process_csv`**: simple file loader that reads a CSV or Excel file into a pandas DataFrame and returns a status message.

3. **`get_dataframe_patterns`**: regex-based pattern fingerprinting function. For every cell in the DataFrame, it converts the value into a shape string where digits become `N`, letter sequences become `W`, and punctuation is preserved. The result is a per-column frequency map of patterns. For example, a date column might produce `{"NN/NN/NNNN": 990, "NN-NN-NNNN": 10}`, immediately revealing a format inconsistency. This utility is used by the Consistency Validator and the Anomaly Detector.

### 1.1: `Outputs` Helper Class

The `Outputs` class wraps raw LLM responses and provides three extraction methods:
- `get_text()`: returns clean text, stripping markdown code fences and handling both string and list-of-blocks response formats (the latter is used by thinking models like Gemma)
- `get_json_obj()`: extracts and parses the first JSON object or array found in the response, which is essential for agents that ask the LLM to return structured data (e.g., lists of column names)
- `get_list_out()`: a convenience alias for parsing a JSON list from the full text

In [2]:
class Outputs:
    """Wraps an LLM response and provides clean text / JSON extraction."""

    def __init__(self, response):
        self.response = response

    def __str__(self):
        return self.get_text()

    def get_list_out(self):
        return json.loads(self.get_text())

    def get_text(self):
        content = self.response
        text = ""

        # Handle the list structure from thinking models (e.g., Gemma)
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and block.get("type") == "text":
                    text = block.get("text", "")
                    break
            if not text and len(content) > 0:
                text = str(content[0])
        else:
            text = str(content)

        # Strip markdown code fences
        text = re.sub(r'```[a-zA-Z]*\n?', '', text)
        text = re.sub(r'\n?```', '', text)
        return text.strip()

    def get_json_obj(self):
        """Extract and parse the first JSON object or array from the response."""
        text = self.get_text()
        match = re.search(r'(\[.*\]|\{.*\})', text, re.DOTALL)
        if match:
            text = match.group(1)
        return json.loads(text)

### 1.2: `process_csv`: File Loading

This function takes a file path string and returns a tuple `[status_message, DataFrame]`. It supports both `.csv` and `.xlsx` formats. If the file cannot be loaded, it returns `None` as the DataFrame so the pipeline can handle the error.

In [3]:
def process_csv(file_path):
    """Load a CSV or Excel file into a pandas DataFrame."""
    if file_path.endswith('.csv'):
        try:
            df = pd.read_csv(file_path)
            return [f"SUCCESS: Loaded CSV — {len(df)} rows, {len(df.columns)} columns: {list(df.columns)}", df]
        except Exception as e:
            return [f"FAILURE: {e}", None]
    elif file_path.endswith('.xlsx'):
        try:
            df = pd.read_excel(file_path)
            return [f"SUCCESS: Loaded Excel — {len(df)} rows, {len(df.columns)} columns: {list(df.columns)}", df]
        except Exception as e:
            return [f"FAILURE: {e}", None]
    return ["FAILURE: Unsupported file type. Please provide a .csv or .xlsx file.", None]

### 1.3: `get_dataframe_patterns`: Regex Pattern Fingerprinting

This is a key utility shared across multiple agents. For every cell in the DataFrame, it converts the value into a *shape string*:
- All digit sequences: `N` (each digit individually)
- All letter sequences: `W`
- Punctuation and separators are preserved as-is
- Null/missing values are mapped to the string `"NULL"`

The result is a dictionary mapping each column name to a frequency count of its patterns. This enables the agents to reason about format consistency and column types without hard-coding expected formats.

In [4]:
def get_dataframe_patterns(df):
    """Build a regex-based pattern frequency map for every column."""

    def get_shape(value):
        if value is pd.NA or pd.isna(value) or value is None:
            return "NULL"
        s = str(value)
        s = re.sub(r'[a-zA-Z]+', 'W', s)
        s = re.sub(r'[0-9]', 'N', s)
        return s

    all_column_patterns = {}
    for col in df.columns:
        patterns = df[col].map(get_shape).value_counts().to_dict()
        all_column_patterns[col] = patterns
    return all_column_patterns

## 2: Agent Class Definitions

The system uses a **Supervisor architecture**: a central orchestrator coordinates four specialised agents in sequence. Each agent class encapsulates its LLM backend and the specific prompts and statistical logic needed for its task.

The agents are:
1. **DataOrchestrator**: parses user intent and loads the dataset
2. **SchemaValidator**: checks data types against regex patterns and validates naming conventions; **automatically corrects** both
3. **CompletenessAnalyst**: detects placeholders via LLM, computes missing-value rates per column and per row, flags sparse columns/rows
4. **ConsistencyValidator**: detects duplicates (exact + key-column), checks format consistency via pattern fingerprints, validates cross-column logic
5. **AnomalyDetector**: LLM-driven column selection → 3σ univariate outlier detection + categorical rare-value detection
6. **RemediatorAgent**: aggregates all findings and produces a final Human-in-the-Loop Action Plan with a Data Reliability Score (0–100)

### 2.1: DataOrchestrator

The orchestrator is the entry point of the pipeline. It receives the user's text input and a file path, then uses the LLM to determine whether the user wants to load and check the file. If yes, it calls `process_csv` to load the data.

In the Streamlit app, this agent handles the interactive file upload and chat. In the notebook, we call its loading logic directly since the intent is always to analyse the dataset.

In [5]:
class DataOrchestrator:
    """Entry point: intent detection + CSV loading."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=0)

    def run_loading(self, user_input, file_path):
        prompt = (
            f"The user says: '{user_input}'. "
            f"They provided a file path: '{file_path}'. "
            "Task: Does the user want me to load and check this file? Answer only 'YES' or 'NO'."
        )
        intent_check = Outputs(self.model.invoke(prompt))
        intent_check = str(intent_check).strip().upper()

        if "YES" in intent_check and file_path:
            result = process_csv(file_path)
            return [result[0], result[1], file_path]
        else:
            return "I'm just a chatbot to check for data quality! I can't really help with anything else, sorry :("

orchestrator = DataOrchestrator()
print("DataOrchestrator initialised.")

DataOrchestrator initialised.


### 2.2: SchemaValidator

The Schema Validator performs two checks and two automatic corrections:

1. **Data Type Validation** (`run_validation_check`): sends the current dtypes and the regex pattern report to the LLM. The LLM compares the dominant patterns (e.g., `NN.NNN`) against the current pandas dtype (e.g., `object`) and flags mismatches. This is pattern-driven, not name-driven, to avoid assumptions.

2. **Data Type Correction** (`run_validation_correction`): the LLM produces a JSON dictionary of `{column: target_type}` corrections. The code then applies type casting with robust cleaning: for integers, it extracts digit sequences; for floats, it handles decimal separators. `errors='coerce'` ensures non-convertible values become `NaN` rather than crashing.

3. **Naming Convention Check** (`run_naming_check`): sends column names to the LLM and asks it to flag violations against standard conventions (special characters, inconsistent casing, leading digits, reserved words).

4. **Naming Correction** (`run_naming_correction`): the LLM produces a JSON dictionary of `{old_name: new_name}` corrections. A deduplication step ensures no naming collisions.

The model used is **Gemini 3.1 Flash Lite Preview**, chosen for its speed and reliability on structured extraction tasks.

In [6]:
class SchemaValidator:
    """Agent 1: Data type + naming convention validation and correction."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

    def run_validation_check(self, df, dataset_name, pattern_report):
        """Compare dtypes against regex patterns to find type mismatches."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"Current Schema (Column Name: Type): {df.dtypes.to_dict()}\n"
            f"Regex Pattern Report (Dominant shapes in data): {pattern_report}\n\n"
            "TASK:\n"
            "You are an Expert Data Analyst. Compare the Current Schema to the Regex Pattern Report "
            "Base the analysis only on the dominant regex patterns for each column, ignoring the column names.\n"
            "to identify necessary type conversions. Apply these rules strictly in order:\n\n"
            "1. **Object to Numeric**: If a column's type is 'object' (dtype 'O') but the regex patterns "
            "show it consists mostly of numbers (e.g., 'N', 'NN.N', 'N.NNN'), flag it to be cast as 'float' or 'int'.\n"
            "2. **String to Numeric**: If a column is already typed as 'string' but contains mostly numeric "
            "patterns (e.g., 'N', 'N.N'), flag it to be cast as 'float' or 'int'.\n\n"
            "IMPORTANT: Flag all columns that meet these criteria, regardless of current formatting errors. "
            "Python will handle the cleaning; you only need to identify the TARGET type.\n\n"
            "OUTPUT RULES:\n"
            "1. If mismatches are found: Begin with exactly: 'Here's a list of columns with type mismatches I found:'\n"
            "   - List each column with a bullet point.\n"
            "   - Provide a brief explanation. Example: '- spesa: Currently an object, but regex shows numeric patterns; should be a float.'\n"
            "2. If all columns correctly match their regex patterns: Return ONLY 'All columns have the expected types they should have.'"
        )
        return self.model.invoke(prompt).content

    def run_validation_correction(self, validation_check, df):
        """Apply type corrections based on the validation report."""
        prompt = (
            f"Context: You are a data correction agent. You are given the following validation report "
            f"that highlights potential issues with the schema of a dataset: {validation_check}\n\n"
            "Task: For each column that is highlighted in the report, populate a json style list with "
            "the type that the columns should be cast into. Only populate the list for the columns that "
            "are highlighted in the report, and leave the other columns out of the list. "
            'Example of expected output if "price" and "id" were highlighted: { "price": "float", "id": "string" }\n\n'
        )
        message = self.model.invoke(prompt)
        corrections = Outputs(message.content).get_json_obj()
        for col, new_type in corrections.items():
            if col not in df.columns:
                continue
            if new_type in ["int", "Int64", "int64", "integer"]:
                cleaned_series = df[col].astype(str).str.extract(r'(\d+)')[0]
                df[col] = pd.to_numeric(cleaned_series, errors="coerce")
                df[col] = df[col].fillna(pd.NA)
                df[col] = df[col].astype("Int64")
            if new_type in ["float", "Float64", "float64"]:
                cleaned_series = df[col].astype(str).str.extract(r'(-?\d+[\d.,]*)')[0]
                df[col] = pd.to_numeric(cleaned_series, errors='coerce')
                df[col] = df[col].fillna(pd.NA)
                df[col] = df[col].astype("float64")
        return df

    def run_naming_check(self, df, dataset_name):
        """Identify column names that violate naming conventions."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"Current column names: {df.columns.to_list()}\n\n"
            "Task: Identify columns where the name of the column doesn't follow a known standard naming "
            "standard or the general structure of the names of the other columns "
            "(e.g., a column using special characters, wrong casing, or reserved words).\n\n"
            "Output Rules:\n"
            "1. If issues found: Begin the report with the phrase:\n"
            "'Here's a list of columns with naming violations I found:'\n"
            "and then list each column using a bullet point, followed by a brief, "
            "natural explanation of the naming violation. Example: '- price: Currently uses uppercase, but should be lowercase.'\n"
            "2. If no issues: Return ONLY 'No columns with naming violations found.'"
        )
        return self.model.invoke(prompt).content

    def run_naming_correction(self, naming_check, df):
        """Apply naming corrections based on the naming report."""
        prompt = (
            f"Context: You are a data correction agent. You are given the following naming validation report "
            f"that highlights potential issues with the column names of a dataset: {naming_check}\n\n"
            'Task: For each column that is highlighted in the report, populate a json style dictionary with '
            'the corrected name that the column should be renamed to. Only populate the dictionary with the '
            'columns that are highlighted in the report, and leave the other columns out of the list. '
            'Example of expected output if "Price" and "id" were highlighted: { "Price": "price", "id": "ID" }\n\n'
        )
        message = self.model.invoke(prompt)
        corrections = Outputs(message.content).get_json_obj()
        safe_corrections = {}
        taken_names = df.columns.to_list()
        for old_name, new_name in corrections.items():
            final_new_name = new_name
            while final_new_name in taken_names:
                final_new_name += "_dup"
            safe_corrections[old_name] = final_new_name
            taken_names.append(final_new_name)
        df = df.rename(columns=safe_corrections)
        return df

schema_validator = SchemaValidator()
print("SchemaValidator initialised.")

SchemaValidator initialised.


### 2.3: CompletenessAnalyst

The Completeness Analyst works in four steps:

1. **Placeholder Detection** (`run_completeness_analysis`): the LLM inspects the head of the DataFrame and returns a list of common placeholder strings (e.g., `"-"`, `"null"`, `"N/A"`, `"unknown"`, and equivalents in the dataset's inferred language). These are then replaced with `pd.NA` in string columns.

2. **Column Missing-Value Computation** (`NA_percentages_columns`): a deterministic function that computes the percentage of `NaN` values per column. Columns exceeding 50% missing are flagged as candidates for removal.

3. **Row Missing-Value Computation** (`NA_percentages_rows`): after excluding the columns already flagged for removal, computes the percentage of missing values per row. Rows exceeding 50% missing are flagged.

4. **Summarisation** (`summarize_columns`, `summarize_rows`): the LLM receives the statistics and produces human-readable reports.

The model used is **Gemma 4 31B IT**, chosen for its strong Italian language understanding (relevant for placeholder detection in Italian datasets) and structured output capabilities.

In [7]:
class CompletenessAnalyst:
    """Agent 2: Missing value detection, placeholder replacement, completeness reporting."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=0)

    def run_completeness_analysis(self, df, dataset_name):
        """Detect placeholder strings via LLM and replace them with pd.NA in string columns."""
        prompt = (
            "You are a data completeness analyst. Your task is to analyze the completeness of a dataset "
            "and provide insights on the missing values. "
            "You are highly competent in your sector and are an expert in identifying common placeholder "
            "strings that are often used to represent missing values in datasets.\n"
            f"Context: {dataset_name}\n"
            f"This is the list of the columns in the dataframe: {list(df.columns)}\n"
            f"This is the head of the dataframe: {df.head().to_dict()}\n"
            "Identify common placeholder strings in the dataframe (like 'null', 'n/a', '-', 'none'), "
            "both in english and the inferred language of the dataset\n"
            "that should be treated as missing values (NaN).\n"
            'Output Rules: Return ONLY a python list of strings. Example: ["-", "null", "N/A", "missing", "none", "n/a", "unknown", "//", "/"]'
        )
        message = self.model.invoke(prompt)
        placeholders = Outputs(message.content).get_json_obj()
        for col in df.columns:
            if df[col].dtype == "string":
                df[col] = df[col].replace(placeholders, pd.NA)
        return df, placeholders

    def summarize_columns(self, schema, overall, to_drop, dataset_name):
        """Ask the LLM to produce a human-readable column completeness report."""
        prompt = (
            f"Context: {dataset_name}\n"
            "The user gave a .csv that has been analysed for completeness\n"
            f"This is the schema of the missing value percentages of all the columns: {schema}\n"
            f"This is the percentage of overall missing values in the entire dataset: {overall}\n"
            f"This is a list of all the columns where the percentage of missing values is above 50%: {to_drop}\n"
            "Task: Summarise the findings of the analysis in a human readable way. Divide it in three sections:\n"
            "1: Overall Completeness Summary: Briefly explain the overall completeness of the dataset based on the overall missing percentage. Use bullet points.\n"
            "2: Column Completeness Highlights: Create a table showing the missing percentage of each column. List critical insights as a bullet point after the table.\n"
            "3: Flag columns for removal: Based on the list of columns with more than 50% missing values, create a bullet point list of the columns that should be removed, along with a brief explanation of why."
        )
        return self.model.invoke(prompt).content

    def summarize_rows(self, schema, to_drop, dataset_name):
        """Ask the LLM to produce a human-readable row completeness report."""
        total_rows = len(schema)
        rows_above_50 = len(to_drop)
        pct_above_50 = rows_above_50 / total_rows if total_rows > 0 else 0
        sample_rows = to_drop[:20]

        prompt = (
            f"Context: {dataset_name}\n"
            f"Total rows: {total_rows}\n"
            f"Rows with >50% missing values: {rows_above_50} ({pct_above_50:.2%})\n"
            f"Sample of problematic row indices (up to 20): {sample_rows}\n"
            "Task: Summarise the findings in a human readable way.\n"
            "4: Row completeness summary: Give a brief summary including the percentage "
            "of rows with more than 50% missing values.\n"
        )
        return self.model.invoke(prompt).content

    @staticmethod
    def NA_percentages_columns(df):
        """Compute per-column missing-value percentages."""
        return {
            col: value
            for col, value in zip(list(df.columns), [x / len(df) for x in list(df.isnull().sum())])
        }

    @staticmethod
    def NA_percentages_rows(df, list_of_droppable_columns):
        """Compute per-row missing-value percentages (excluding columns flagged for removal)."""
        df_filtered = df.drop(columns=list_of_droppable_columns)
        return {
            idx: value
            for idx, value in zip(
                list(df.index),
                [x / len(df_filtered.columns) for x in list(df_filtered.isnull().sum(axis=1))]
            )
        }

completeness_analyst = CompletenessAnalyst()
print("CompletenessAnalyst initialised.")

CompletenessAnalyst initialised.


### 2.4: ConsistencyValidator

The Consistency Validator performs three checks:

1. **Duplicate Detection** (`run_duplicate_detection`): counts exact row duplicates and removes them in-place. The LLM also identifies likely unique-identifier columns (e.g., `_id`, `tax_code`), and we check for duplicates on those keys specifically.

2. **Format Consistency** (`run_format_consistency_check`): uses the regex pattern fingerprint to flag columns where a small minority of values deviate from the dominant format. The LLM is instructed to only flag columns with a clear dominant format broken by outliers, and to ignore high-variety columns like descriptions.

3. **Cross-Column Logic** (`run_cross_column_logic`): sends a sample of 30 rows (cleaned of sparse columns and null rows) to the LLM. The LLM infers logical rules between columns (e.g., date ordering, mathematical sums, redundant mirror columns) and checks whether they hold.

The model used is **Gemini 3.1 Flash Lite Preview** for fast structured extraction.

In [8]:
class ConsistencyValidator:
    """Agent 3: Duplicate detection, format consistency, cross-column logic."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

    def run_duplicate_detection(self, df, dataset_name):
        """Detect exact duplicates and key-column duplicates; remove exact dupes in-place."""
        exact_dupes = df.duplicated().sum()
        prompt = (
            f"This is the dataset name: {dataset_name}\n"
            f"Here are the columns of the dataset: {list(df.columns)}\n"
            "Task: Identify any columns that likely represent a unique identifier for a record\n"
            '(e.g., "ID", "Tax Code", "Email", "SSN", "SKU").\n'
            'Output Rules: Return ONLY a python list of strings containing the exact column names. '
            'Example: ["user_id", "email"]\n'
            "If none are found, return an empty list []."
        )
        message = self.model.invoke(prompt)
        key_columns = Outputs(message.content).get_json_obj()

        key_dupes_report = []
        if key_columns:
            for col in key_columns:
                if col in df.columns:
                    dupes = df.duplicated(subset=[col]).sum()
                    if dupes > 0:
                        key_dupes_report.append(f"Column '{col}' has {dupes} duplicate entries.")

        df.drop_duplicates(inplace=True)
        return {
            "exact_duplicates": exact_dupes,
            "key_column_duplicates": key_dupes_report if key_dupes_report else "No key duplicates found."
        }

    def run_format_consistency_check(self, pattern_report_str, df, dataset_name):
        """Flag columns with inconsistent value formats based on pattern fingerprints."""
        prompt = (
            f"This is the dataset name: {dataset_name}\n"
            f"Here are the columns of the dataset: {list(df.columns)}\n"
            f"You are given the following regex report on the columns of the dataframe: {pattern_report_str}\n\n"
            "TASK: Flag a column for inconsistent patterns and list how many pattern violations are there "
            "in that column compared to the dominant pattern.\n"
            "Anomalous patterns are defined as patterns that appear in less than 0.05 percent of the rows "
            "compared to a dominant pattern.\n"
            "This is just a general guideline and should be ignored in favor of following the rules if a contradiction is found.\n\n"
            "RULES for flagging:\n"
            "- Do NOT flag a column if it has a high variety of patterns (e.g., a 'Description' column).\n"
            "- ONLY flag a column if there is a clear 'Standard Format' being broken by a few outliers.\n"
            "- Example: If 'Date' has 990 'N-N-N' and 10 'N/N/N', flag 'N/N/N' as the inconsistency.\n\n"
            "Output Rules:\n"
            "1. If format inconsistencies are found: List the column name and briefly explain the mixed formats.\n"
            "2. If formatting looks consistent across all columns: Return ONLY 'Formats are consistent'."
        )
        return self.model.invoke(prompt).content

    def run_cross_column_logic(self, df_modified):
        """Infer and validate logical rules between columns on a sample."""
        sample_size = min(30, len(df_modified))
        sample_data = df_modified.sample(n=sample_size, random_state=42).to_markdown()
        prompt = (
            f"You are a Senior Data Quality Analyst. I am providing a sample of {sample_size} rows "
            f"from a larger dataset.\n\n"
            f"### DATA SAMPLE:\n{sample_data}\n\n"
            "### TASK:\n"
            "1. **Analyze Relationships**: Based on the column names and the sample, identify logical rules "
            "that *should* exist (e.g., Date order, Mathematical sums, or Redundant columns).\n"
            "2. **Audit the Sample**: Check the provided rows against these rules.\n"
            "3. **Identify Redundancy**: Specifically look for columns that appear to be 'mirrors' of "
            "each other (identical data under different names).\n\n"
            "### OUTPUT FORMAT (Strict Markdown):\n"
            "If issues are found, return a report with these sections:\n"
            "**Inferred Logical Rules:**\n"
            "- [Rule description]\n\n"
            "**Data Redundancy Alerts:**\n"
            "- [Column A] and [Column B] appear to contain identical information.\n\n"
            "If NO logical violations or rules are found, return ONLY: 'No logical violations detected in sample.'"
            "If you do mention any specific rows in the report, make sure to remind the user that those "
            "specific rows are just examples and that the issues may be present in other rows as well."
        )
        return self.model.invoke(prompt).content

consistency_validator = ConsistencyValidator()
print("ConsistencyValidator initialised.")

ConsistencyValidator initialised.


### 2.5: AnomalyDetector

The Anomaly Detector performs two types of detection:

1. **Univariate Outlier Detection** (`univariate_outlier_detection`): the LLM selects numerical columns that are semantically meaningful for outlier analysis (e.g., `spesa` but not `ente`). For each selected column, the **3σ rule** is applied: values more than 3 standard deviations from the mean are flagged. If more than 10 outliers are found in a column, detailed values are written to a separate text report (to keep the in-notebook output readable).

2. **Categorical Anomaly Detection** (`categorical_outlier_detection`): the LLM selects categorical columns suitable for frequency analysis (e.g., `tipo_imposta` but not `descrizione`). Values appearing in fewer than 1% of rows are flagged as rare/anomalous categories.

The model used is **Gemma 4 31B IT**. The `pd.to_numeric(..., errors='coerce')` call in the univariate detection acts as a failsafe in case the Schema Validator's type casting missed some non-numeric values.

In [9]:
class AnomalyDetector:
    """Agent 4: Univariate outlier detection + categorical rare-value detection."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=0)

    def univariate_outlier_detection(self, pattern_report_str, df, dataset_name):
        """LLM selects numerical columns, then 3-sigma outlier detection is applied."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"This is a sample of the dataframe you are working on: {df.head()}\n"
            f"This is the regex pattern report of all the columns of the dataframe: {pattern_report_str}\n"
            "Task: Identify candidate columns for univariate outlier detection.\n"
            "A column is a candidate for univariate outlier detection if both of these conditions are true:\n"
            "1. The pattern report of a column shows that most of the entries are made up of numbers "
            "(e.g. the vast majority are in the pattern NN.NNN, N.N, NN.N)\n"
            "2. It makes sense to check for numerical outliers in that column using standard deviations "
            "(e.g. it would make sense to check for outliers in a column called 'expenses', but not in a column called 'id')\n"
            'Output: Return ONLY a python list in JSON style of strings. Example: ["rate", "spesa"]'
        )
        message = self.model.invoke(prompt)
        univariate_columns = Outputs(message.content).get_json_obj()
        report_to_disk = None
        report = ""

        if len(univariate_columns) == 0:
            report += "No columns were found as candidates for univariate outliers detection.\n"
        else:
            for col in univariate_columns:
                if col not in df.columns:
                    continue
                series = pd.to_numeric(df[col], errors='coerce').dropna()
                if series.empty:
                    continue
                mean_val = series.mean()
                std_val = series.std()
                upper_limit = mean_val + (3 * std_val)
                lower_limit = mean_val - (3 * std_val)
                outlier_mask = (series > upper_limit) | (series < lower_limit)
                outlier_data = series[outlier_mask]

                if len(outlier_data) == 0:
                    report += f"Column '{col}': No outliers detected.\n\n"
                else:
                    report += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                    if len(outlier_data) < 10:
                        report += f"Details: {outlier_data.to_dict()}\n\n"
                    if len(outlier_data) >= 10 and not report_to_disk:
                        report += "Details: Too many outliers to display (more than 10), check the final txt report for more details.\n\n"
                        report_to_disk = "Univariate outliers: \n\n"
                        report_to_disk += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                        report_to_disk += f"Details: {outlier_data.to_dict()}\n\n"
                    elif len(outlier_data) >= 10 and report_to_disk:
                        report += "Details: Too many outliers to display (more than 10), check the final txt report for more details.\n\n"
                        report_to_disk += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                        report_to_disk += f"Details: {outlier_data.to_dict()}\n\n"

            report += ("Remember that this analysis is done only on values that could be converted into a number. "
                       "Refer to the report for the list of rows and values that cannot be cast into a number.\n"
                       "All the values that couldn't be cast have been ignored.")

        return [report, report_to_disk]

    def categorical_outlier_detection(self, pattern_report_str, df, dataset_name, outlier_report_file):
        """LLM selects categorical columns, then rare-value detection is applied."""
        prompt = (
            f"Context: {dataset_name}\n"
            f"This is a sample of the dataframe you are working on: {df.head()}\n"
            f"This is the regex pattern report of all the columns of the dataframe: {pattern_report_str}\n"
            "Task: Identify candidate columns for categorical outlier detection.\n"
            "A column is a candidate for categorical outlier detection if both of these conditions are true:\n"
            "1. The pattern report of a column shows that most of the entries are made up of words "
            "(e.g. the vast majority are in the pattern W W, W, W W W)\n"
            "2. It makes sense to check for categorical outliers in that column (e.g. it would make sense "
            "to check for outliers in a column called 'job title' or 'education level', but not in a "
            "column called 'notes' or 'client names')\n"
            'Output: Return ONLY a python list in JSON style of strings. Example: ["job title", "level"]'
        )
        message = self.model.invoke(prompt)
        categorical_columns = Outputs(message.content).get_json_obj()
        report_to_disk = outlier_report_file
        report = ""
        first_iteration = False

        if len(categorical_columns) == 0:
            report += "No columns were found as candidates for categorical outliers detection.\n"
        else:
            for col in categorical_columns:
                if col not in df.columns:
                    continue
                series = df[col].astype(str)
                if series.empty:
                    continue
                series_counts = series.value_counts()
                outlier_mask = (series_counts < len(series) * 0.01)
                outlier_data = series_counts[outlier_mask]

                if len(outlier_data) == 0:
                    report += f"Column '{col}': No outliers detected.\n\n"
                else:
                    report += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                    if len(outlier_data) < 10:
                        report += f"Details: {outlier_data.to_dict()}\n\n"
                    if len(outlier_data) >= 10 and not report_to_disk:
                        report += "Details: Too many outliers to display (more than 10), check the final txt report for more details.\n\n"
                        report_to_disk = "Categorical outliers: \n\n"
                        report_to_disk += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                        report_to_disk += f"Details: {outlier_data.to_dict()}\n\n"
                    elif len(outlier_data) >= 10 and report_to_disk:
                        report += "Details: Too many outliers to display (more than 10), check the final txt report for more details.\n\n"
                        if not first_iteration:
                            report_to_disk += "Categorical outliers: \n\n"
                            first_iteration = True
                        report_to_disk += f"Column '{col}': Found {len(outlier_data)} outliers.\n"
                        report_to_disk += f"Details: {outlier_data.to_dict()}\n\n"

        return [report, report_to_disk]

anomaly_detector = AnomalyDetector()
print("AnomalyDetector initialised.")

AnomalyDetector initialised.


### 2.6: RemediatorAgent

The Remediator is the final agent in the pipeline. It receives the complete reports from the Completeness Analyst, Consistency Validator, and Anomaly Detector, and produces:

1. **A Data Reliability Score** (0–100) with justification: a single metric that summarises the overall quality of the dataset
2. **A Human-in-the-Loop Action Plan**: specific, actionable steps the data engineering team should take to finish cleaning the dataset, covering missing values (imputation strategies), outliers/anomalies (cap, delete, or investigate), and inconsistencies (format fixes, cross-column logic resolution)

This agent is designed to bridge the gap between automated detection and human decision-making: the pipeline identifies issues, but the final resolution is left to domain experts.

The model used is **Gemma 4 31B IT** for its strong reasoning and report-writing capabilities.

In [10]:
class RemediatorAgent:
    """Agent 5: Aggregates all findings into a final action plan + reliability score."""

    def __init__(self):
        self.model = ChatGoogleGenerativeAI(model="gemma-4-31b-it", temperature=0)

    def generate_remediation_report(self, completeness_report, consistency_report, anomaly_report, dataset_name):
        """Generate final Human-in-the-Loop Action Plan and Data Reliability Score."""
        prompt = (
            f"You are a Lead Data Quality Engineer. I am providing you with three diagnostic reports "
            f"for a dataset named '{dataset_name}'.\n\n"
            "Note: The pipeline has ALREADY standardized column names, casted data types, replaced placeholders with NaNs, "
            "and removed exact duplicate rows.\n\n"
            f"### 1. Completeness Report:\n{completeness_report}\n\n"
            f"### 2. Consistency Report:\n{consistency_report}\n\n"
            f"### 3. Anomaly Report:\n{anomaly_report}\n\n"
            "### TASK:\n"
            "Generate a final 'Human-in-the-Loop Action Plan' and a 'Data Reliability Score'.\n\n"
            "### OUTPUT FORMAT (Strict Markdown):\n"
            "## Final Data Reliability Score: [Insert Score 0-100]/100\n"
            "**Justification:** [1-2 sentences explaining why this score was given based on the severity of remaining issues.]\n\n"
            "## Human-in-the-Loop Action Plan\n"
            "Based on the reports, here are the exact steps the data engineering team should take to finish cleaning this dataset:\n"
            "- **Missing Values:** [Provide specific imputation strategies (e.g., median, mean, mode) for remaining missing values].\n"
            "- **Outliers & Anomalies:** [Advise on what to do with the specific outliers flagged—should they be capped, deleted, or investigated?]\n"
            "- **Inconsistencies:** [Advise on how to manually resolve any format or cross-column logic issues found].\n"
        )
        return self.model.invoke(prompt).content

remediator = RemediatorAgent()
print("RemediatorAgent initialised.")

RemediatorAgent initialised.


## 3: Experiment 1: Full Pipeline on `spesa.csv`

We now run the complete AGENTS4DQ pipeline on the first dataset: `spesa.csv`, a NoiPA expenditure file with 7,543 rows and 18 columns covering tax categories, ministry descriptions, and spending amounts.

The dataset contains intentional noise for testing purposes, including:
- Redundant/mirror columns (`spesa` vs `SPESA TOTALE`, `cod_imposta` vs `2cod_imposta` vs `cod imposta ext`)
- Naming convention violations (`ente%code`, `2cod_imposta`, `cod imposta ext`, `Tipo Imposta`)
- Columns with high percentages of missing values (`note`, `fonte_dato`)
- Mixed data types (numeric columns stored as strings)

### Load Dataset

We load the CSV and display basic information: row count, column list, first 5 rows, data types, and null counts. This gives us a baseline understanding of what the agents will be working with.

In [11]:
DATA_PATH_1 = "data/raw/spesa.csv"

status, df1 = process_csv(DATA_PATH_1)
print(status)
df1.head()

SUCCESS: Loaded CSV — 7543 rows, 18 columns: ['_id', 'rata', 'ente', 'descrizione', 'cod_tipoimposta', 'tipo_imposta', 'cod_imposta', 'imposta', 'spesa', 'aggregation-time', 'area_geografica', 'note', 'fonte_dato', 'Tipo Imposta', 'SPESA TOTALE', '2cod_imposta', 'cod imposta ext', 'ente%code']


,_id,rata,ente,descrizione,cod_tipoimposta,tipo_imposta,cod_imposta,imposta,spesa,aggregation-time,area_geografica,note,fonte_dato,Tipo Imposta,SPESA TOTALE,2cod_imposta,cod imposta ext,ente%code
0,65ee5ac5f458af56d2af532f,202402,867,AGENZIA ITALIANA DEL FARMACO - AIFA,2,Erariali,6,IRAP,182904.47999999954,2024-03-11T02:01:04.421,Nord,NaN,NaN,Erariali,182904.47999999954,6,6,867
1,668f34120377f62206882cd8,202406,921,A.O. S. GIOVANNI ADDOLORATA,3,Previdenziali,13,Previdenziali a carico del datore di lavoro,2110811.34,2024-07-11T03:01:16.866,Nord,NaN,NaN,Previdenziali,2110811.34,13,13,921
2,66e0ee64f458af54a5dbc7fa,202408,9,MINISTERO DELLA GIUSTIZIA,4,Varie,10,Ritenute Sindacali,732614.36,2024-09-11T03:01:11.704,Nord,NaN,NaN,Varie,732614.36,10,10,9
3,663ec5eb3f62190222cfb590,202404,12,MINISTERO DELL'INTERNO,2,Erariali,6,IRAP,43365008.73,2024-05-11T03:01:07.269,Nord,NaN,NaN,Erariali,43365008.73,6,6,12
4,6731590c568f146446645d59,202410,910,AZIENDA SANITARIA LOCALE ROMA 6,2,Erariali,6,IRAP,1310915.22,2024-11-11T02:00:28.485,Isole,NaN,NaN,Erariali,1310915.22,6,6,910


Below we inspect the data types and null counts to understand the starting state of the dataset before any agent intervention.

In [12]:
print("Shape:", df1.shape)
print("\nDtypes:")
print(df1.dtypes)
print("\nNull counts:")
print(df1.isnull().sum())

Shape: (7543, 18)

Dtypes:
_id                 object
rata                object
ente                object
descrizione         object
cod_tipoimposta      int64
tipo_imposta        object
cod_imposta         object
imposta             object
spesa               object
aggregation-time    object
area_geografica     object
note                object
fonte_dato          object
Tipo Imposta        object
SPESA TOTALE        object
2cod_imposta         int64
cod imposta ext      int64
ente%code            int64
dtype: object

Null counts:
_id                    0
rata                   0
ente                 118
descrizione          169
cod_tipoimposta        0
tipo_imposta           0
cod_imposta           72
imposta              137
spesa                  0
aggregation-time       0
area_geografica     1582
note                7393
fonte_dato          7468
Tipo Imposta           0
SPESA TOTALE           0
2cod_imposta           0
cod imposta ext        0
ente%code              0
dtype: in

### Step 1: Schema Validation

We first compute the regex pattern fingerprint of the dataset (needed by the Schema Validator to compare against dtypes), then run the data type validation and naming convention checks. After each check, the agent automatically applies corrections.

We expect the validator to flag:
- Type mismatches: columns like `spesa` or `cod_imposta` that are stored as strings/objects but contain mostly numeric patterns
- Naming violations: `ente%code` (special character), `2cod_imposta` (leading digit), `cod imposta ext` (spaces), `Tipo Imposta` and `SPESA TOTALE` (inconsistent casing)

In [13]:
pattern_report_1 = get_dataframe_patterns(df1)
pattern_report_1_str = json.dumps(pattern_report_1, indent=4)

print("DATA TYPE VALIDATION")
type_check = schema_validator.run_validation_check(df1, DATA_PATH_1, pattern_report_1_str)
type_check_text = Outputs(type_check).get_text()
print(type_check_text)

DATA TYPE VALIDATION
Here's a list of columns with type mismatches I found:

- **rata**: Currently an object, but regex patterns (e.g., 'NNNNNN', 'NNNN-NN') suggest it contains numeric or date-like structures; should be converted to a numeric or datetime type.
- **ente**: Currently an object, but regex patterns (e.g., 'NNN', 'NN', 'N') indicate numeric identifiers; should be cast as an integer.
- **cod_imposta**: Currently an object, but regex patterns (e.g., 'NN', 'N') indicate numeric identifiers; should be cast as an integer.
- **imposta**: Currently an object, but regex patterns (e.g., 'W', 'W W') suggest categorical data, but if numeric values are embedded, it should be evaluated for float conversion.
- **spesa**: Currently an object, but regex patterns (e.g., 'NNNNNNN.NN') show numeric currency formats; should be a float.
- **aggregation-time**: Currently an object, but regex patterns (e.g., 'NNNN-NN-NN') indicate date/time structures; should be converted to datetime.
- **SPESA T

The Schema Validator has identified the type mismatches. We now apply automatic corrections: the LLM produces a JSON mapping of `{column: target_type}`, and the code casts each column accordingly. After correction, we print the updated dtypes to confirm the changes.

In [14]:
df1 = schema_validator.run_validation_correction(type_check_text, df1)
print("Type corrections applied. Updated dtypes:")
print(df1.dtypes)

Type corrections applied. Updated dtypes:
_id                  object
rata                 object
ente                  Int64
descrizione          object
cod_tipoimposta       int64
tipo_imposta         object
cod_imposta           Int64
imposta             float64
spesa               float64
aggregation-time     object
area_geografica      object
note                 object
fonte_dato           object
Tipo Imposta         object
SPESA TOTALE        float64
2cod_imposta          int64
cod imposta ext       int64
ente%code             int64
dtype: object


Next, we run the naming convention check. The LLM examines all column names and flags those that don't follow a consistent standard.

In [15]:
print("NAMING CONVENTION CHECK")
naming_check = schema_validator.run_naming_check(df1, DATA_PATH_1)
naming_check_text = Outputs(naming_check).get_text()
print(naming_check_text)

NAMING CONVENTION CHECK
Here's a list of columns with naming violations I found:

- aggregation-time: Uses a hyphen instead of an underscore, which can cause syntax issues in many programming languages.
- Tipo Imposta: Uses title case with a space, inconsistent with the snake_case style of the other columns.
- SPESA TOTALE: Uses uppercase with a space, inconsistent with the snake_case style of the other columns.
- 2cod_imposta: Starts with a number, which is generally discouraged as it can cause issues when referencing the column as an object attribute.
- cod imposta ext: Uses spaces instead of underscores, inconsistent with the snake_case style of the other columns.
- ente%code: Uses a special character (%) which is non-standard and can cause issues in database queries or code.


We now apply the naming corrections automatically. The LLM produces a JSON mapping of `{old_name: new_name}`, and a deduplication step ensures no collisions with existing column names. After correction, we print the updated column list.

In [16]:
df1 = schema_validator.run_naming_correction(naming_check_text, df1)
print("Naming corrections applied. Updated columns:")
print(list(df1.columns))

Naming corrections applied. Updated columns:
['_id', 'rata', 'ente', 'descrizione', 'cod_tipoimposta', 'tipo_imposta', 'cod_imposta', 'imposta', 'spesa', 'aggregation_time', 'area_geografica', 'note', 'fonte_dato', 'tipo_imposta_dup', 'spesa_totale', 'cod_imposta_2', 'cod_imposta_ext', 'ente_code']


### Step 2: Completeness Analysis

The Completeness Analyst first detects placeholder strings that should be treated as missing values, then computes per-column and per-row missing-value percentages, and finally produces a summarised report.

We start with placeholder detection: the LLM examines the head of the DataFrame and returns a list of strings to replace with `pd.NA`.

In [17]:
df1, placeholders_1 = completeness_analyst.run_completeness_analysis(df1, DATA_PATH_1)
print("Placeholders detected and replaced with NaN:", placeholders_1)

Placeholders detected and replaced with NaN: ['null', 'n/a', 'N/A', 'none', 'None', 'nan', 'NaN', 'missing', 'unknown', '-', '_', '/', '//', '...', 'N.D.', 'n.d.', 'non disponibile', 'non pervenuto', 'sconosciuto', 'mancante']


Now we compute the per-column missing-value percentages. Columns with more than 50% missing are flagged as candidates for removal, as they are unlikely to be useful for analysis and could introduce bias if retained.

In [18]:
completeness_cols_1 = completeness_analyst.NA_percentages_columns(df1)
droppable_columns_1 = [col for col, v in completeness_cols_1.items() if v > 0.5]
overall_missing_1 = sum(completeness_cols_1.values()) / len(completeness_cols_1)

comp_df_1 = pd.DataFrame.from_dict(completeness_cols_1, orient="index", columns=["Missing %"])
comp_df_1 = comp_df_1.sort_values("Missing %", ascending=False)
comp_df_1["Missing %"] = comp_df_1["Missing %"].map(lambda x: f"{x:.2%}")
print(comp_df_1.to_string())
print(f"\nOverall missing: {overall_missing_1:.2%}")
print(f"Columns flagged for removal (>50% missing): {droppable_columns_1}")

                 Missing %
imposta            100.00%
fonte_dato          99.01%
note                98.01%
area_geografica     20.97%
ente                 4.00%
cod_imposta          2.98%
descrizione          2.24%
spesa                1.48%
spesa_totale         1.48%
_id                  0.00%
tipo_imposta_dup     0.00%
cod_imposta_ext      0.00%
cod_imposta_2        0.00%
aggregation_time     0.00%
rata                 0.00%
tipo_imposta         0.00%
cod_tipoimposta      0.00%
ente_code            0.00%

Overall missing: 18.34%
Columns flagged for removal (>50% missing): ['imposta', 'note', 'fonte_dato']


Next, we compute per-row completeness. After excluding the columns already flagged for removal, we identify rows where more than 50% of the remaining values are missing.

In [19]:
completeness_rows_1 = completeness_analyst.NA_percentages_rows(df1, droppable_columns_1)
droppable_rows_1 = [key for key, v in completeness_rows_1.items() if v > 0.5]

print(f"Total rows: {len(completeness_rows_1)}")
print(f"Rows with >50% missing (excl. flagged columns): {len(droppable_rows_1)}")
print(f"Percentage: {len(droppable_rows_1)/len(completeness_rows_1):.2%}")

Total rows: 7543
Rows with >50% missing (excl. flagged columns): 0
Percentage: 0.00%


Finally, the LLM produces human-readable summaries of the column and row completeness findings.

In [20]:
summary_cols_1 = Outputs(
    completeness_analyst.summarize_columns(completeness_cols_1, overall_missing_1, droppable_columns_1, DATA_PATH_1)
).get_text()

summary_rows_1 = Outputs(
    completeness_analyst.summarize_rows(completeness_rows_1, droppable_rows_1, DATA_PATH_1)
).get_text()

completeness_report_1 = f"{summary_cols_1}\n\n{summary_rows_1}"
print(completeness_report_1)

### 1: Overall Completeness Summary
* **High General Integrity:** The dataset is exceptionally complete overall, with only **0.18%** of the total data points missing.
* **Reliable Core Data:** The vast majority of the information is intact, suggesting that the primary records are well-maintained.

### 2: Column Completeness Highlights

| Column | Missing Percentage |
| :--- | :--- |
| _id | 0.00% |
| rata | 0.00% |
| ente | 4.00% |
| descrizione | 2.24% |
| cod_tipoimposta | 0.00% |
| tipo_imposta | 0.00% |
| cod_imposta | 2.98% |
| imposta | 100.00% |
| spesa | 1.48% |
| aggregation_time | 0.00% |
| area_geografica | 20.97% |
| note | 98.01% |
| fonte_dato | 99.01% |
| tipo_imposta_dup | 0.00% |
| spesa_totale | 1.48% |
| cod_imposta_2 | 0.00% |
| cod_imposta_ext | 0.00% |
| ente_code | 0.00% |

**Critical Insights:**
* **Perfect Columns:** A significant number of columns (e.g., `_id`, `rata`, `ente_code`) have 0% missing values, providing a solid foundation for indexing and categoriz

### Step 3: Consistency Validation

The Consistency Validator performs three checks in sequence: duplicate detection, format consistency analysis, and cross-column logic validation.

We start with duplicate detection. The agent counts exact row duplicates, removes them in-place, and also identifies likely key columns to check for key-level duplicates.

In [21]:
duplicate_results_1 = consistency_validator.run_duplicate_detection(df1, DATA_PATH_1)

print(f"Exact row duplicates found (and removed): {duplicate_results_1['exact_duplicates']}")
print(f"Key column duplicates: {duplicate_results_1['key_column_duplicates']}")
print(f"Shape after deduplication: {df1.shape}")

Exact row duplicates found (and removed): 40
Key column duplicates: ["Column '_id' has 65 duplicate entries."]
Shape after deduplication: (7503, 18)


Next, we recompute the pattern fingerprint (since the schema may have changed after type corrections) and run the format consistency check. The LLM flags columns where a dominant pattern is broken by a few outlier formats.

In [22]:
pattern_report_1_updated = get_dataframe_patterns(df1)
pattern_report_1_updated_str = json.dumps(pattern_report_1_updated, indent=4)

format_results_1 = Outputs(
    consistency_validator.run_format_consistency_check(pattern_report_1_updated_str, df1, DATA_PATH_1)
).get_text()
print(format_results_1)

Based on the provided regex report, here are the columns with inconsistent patterns:

**1. rata**
*   **Dominant pattern:** `NNNNNN` (6995 rows)
*   **Inconsistent patterns:**
    *   `NNNN-NN`: 208 violations
    *   `NN/NNNN`: 110 violations
    *   `W NNNN`: 95 violations
    *   `W-NNNN`: 95 violations

**2. aggregation_time**
*   **Dominant pattern:** `NNNN-NN-NNWNN:NN:NN.NNN` (6903 rows)
*   **Inconsistent patterns:**
    *   `NN/NN/NNNN`: 134 violations
    *   `NN-NN-NN`: 130 violations
    *   `NNNN/NN/NN`: 117 violations
    *   `NN.NN.NNNN`: 111 violations
    *   `W NN NNNN`: 108 violations

**3. tipo_imposta**
*   **Dominant pattern:** `W` (7499 rows)
*   **Inconsistent patterns:**
    *   `W W`: 2 violations
    *   `W `: 2 violations

**4. ente**
*   **Dominant pattern:** `NNN.N` (3873 rows)
*   **Inconsistent patterns:**
    *   `NN.N`: 2074 violations
    *   `N.N`: 1049 violations
    *   `NULL`: 301 violations
    *   `NNNN.N`: 206 violations

**5. cod_imposta**
*   

Finally, we run the cross-column logic check. Before sending data to the LLM, we drop the columns flagged as too sparse and remove rows with remaining nulls, to give the LLM a clean 30-row sample to reason about.

In [23]:
df1_clean = df1.drop(columns=droppable_columns_1, errors='ignore').dropna(axis=0)
print(f"Clean sample shape (after dropping sparse cols + null rows): {df1_clean.shape}")

logic_results_1 = Outputs(
    consistency_validator.run_cross_column_logic(df1_clean)
).get_text()
print(logic_results_1)

Clean sample shape (after dropping sparse cols + null rows): (5318, 15)
As a Senior Data Quality Analyst, I have reviewed the provided sample. Below is the audit report based on the observed data patterns.

### Inferred Logical Rules:
- **Mathematical Consistency**: The `spesa` (expenditure) column should logically match the `spesa_totale` column.
- **Data Integrity**: The `spesa` column should contain non-negative values.
- **Standardization**: The `aggregation_time` column should follow a consistent ISO 8601 timestamp format.
- **Categorical Mapping**: `cod_tipoimposta` should have a 1:1 mapping with `tipo_imposta`.
- **Entity Consistency**: `ente` and `ente_code` should contain identical values representing the unique identifier for the entity.

### Data Redundancy Alerts:
- **`ente` and `ente_code`**: These columns appear to contain identical information across all rows.
- **`cod_imposta`, `cod_imposta_2`, and `cod_imposta_ext`**: These three columns appear to be mirrors of each ot

We now assemble the full consistency report for the Remediator.

In [24]:
dupe_text_1 = (
    f"**Exact row duplicates found:** {duplicate_results_1['exact_duplicates']}\n\n"
    f"**Key column duplicates:** {duplicate_results_1['key_column_duplicates']}"
)
consistency_report_1 = "\n\n".join([
    "### Duplicate Analysis Results:",
    dupe_text_1.strip(),
    "### Format Consistency Analysis Results:",
    format_results_1.strip(),
    "### Cross Column Consistency Analysis Results:",
    logic_results_1.strip()
])
print("Consistency report assembled.")

Consistency report assembled.


### Step 4: Anomaly Detection

The Anomaly Detector first runs univariate outlier detection on LLM-selected numerical columns (using the 3σ rule), then runs categorical rare-value detection on LLM-selected categorical columns (flagging values appearing in <1% of rows).

In [25]:
print("UNIVARIATE OUTLIER DETECTION")
outlier_report_1, outlier_file_1 = anomaly_detector.univariate_outlier_detection(
    pattern_report_1_updated_str, df1, DATA_PATH_1)
print(outlier_report_1)

UNIVARIATE OUTLIER DETECTION
Column 'spesa': Found 39 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Column 'spesa_totale': Found 34 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Remember that this analysis is done only on values that could be converted into a number. Refer to the report for the list of rows and values that cannot be cast into a number.
All the values that couldn't be cast have been ignored.


Now we run the categorical anomaly detection. The LLM selects columns where frequency analysis is meaningful, and values appearing in fewer than 1% of rows are flagged.

In [26]:
print("CATEGORICAL ANOMALY DETECTION")
cat_report_1, outlier_file_1 = anomaly_detector.categorical_outlier_detection(
    pattern_report_1_updated_str, df1, DATA_PATH_1, outlier_file_1)
print(cat_report_1)

CATEGORICAL ANOMALY DETECTION
Column 'tipo_imposta': Found 5 outliers.
Details: {'Da definire': 2, 'Erariali ': 2, 'Mista': 2, 'erariali': 1, 'ERARIALI': 1}

Column 'area_geografica': No outliers detected.

Column 'tipo_imposta_dup': No outliers detected.




We assemble the full anomaly detection report for the Remediator.

In [27]:
anomaly_report_1 = "\n\n".join([
    "### Univariate Outlier Detection Results:",
    outlier_report_1.strip(),
    "### Categorical Outlier Detection Results:",
    cat_report_1.strip()
])
print("Anomaly report assembled.")

Anomaly report assembled.


### Step 5: Remediation Report

The Remediator receives all three diagnostic reports and produces a final Data Reliability Score (0–100) with justification, plus a Human-in-the-Loop Action Plan with specific steps for the data engineering team.

In [28]:
remediation_1 = Outputs(
    remediator.generate_remediation_report(
        completeness_report_1, consistency_report_1, anomaly_report_1, DATA_PATH_1
    )
).get_text()
print(remediation_1)

## Final Data Reliability Score: 72/100
**Justification:** While the dataset has high overall completeness, it suffers from critical primary key violations (`_id` duplicates), logical contradictions in financial values (negative expenditure), and significant format noise in key categorical and temporal columns.

## Human-in-the-Loop Action Plan
Based on the reports, here are the exact steps the data engineering team should take to finish cleaning this dataset:

- **Missing Values:** 
    - **Drop Columns:** Immediately remove `imposta`, `fonte_dato`, and `note` as they provide no statistical value.
    - **Imputation:** For `area_geografica` (21% missing), do not use simple mode imputation. Instead, perform a lookup mapping using the `ente_code` to fill missing geographic areas based on the entity's known location. If no mapping exists, label as "Unknown".
    - **Core Data:** `spesa` and `spesa_totale` missing values (1.48%) should be imputed using the median of the respective `tipo_i

## 4: Experiment 2: Full Pipeline on `attivazioniCessazioni.csv`

To validate that the system generalises to structurally different datasets, we now run the full pipeline on a second NoiPA file containing personnel activation/termination records (20,102 rows, 19 columns).

This dataset has different columns (`mese`, `anno`, `attivazioni`, `cessazioni`, `qualifica`, etc.) and different types of intentional noise:
- Naming violations: `3descrizione` (leading digit), `regione%sede` (special character), `att ivazioni` (space in name), `RATA` and `Provincia Sede` and `CODICE ENTE` (inconsistent casing)
- Casing inconsistencies in values: `Aq` vs `AQ` in `provincia_sede`
- Redundant/mirror columns (e.g., `provincia_sede` vs `Provincia Sede`, `codice_ente` vs `CODICE ENTE`)
- High-missing columns (`note`, `fonte_dato`, `qualifica`)

### Load Dataset

We load the second CSV and display the same baseline information.

In [29]:
DATA_PATH_2 = "data/raw/attivazioniCessazioni.csv"

status2, df2 = process_csv(DATA_PATH_2)
print(status2)
df2.head()

SUCCESS: Loaded CSV — 20102 rows, 19 columns: ['_id', 'mese', 'anno', 'codice_ente', 'descrizione_ente', 'provincia_sede', 'regione_sede', 'attivazioni', 'cessazioni', 'RATA', 'aggregation-time', 'qualifica', 'note', 'fonte_dato', 'Provincia Sede', 'CODICE ENTE', '3descrizione', 'regione%sede', 'att ivazioni']


,_id,mese,anno,codice_ente,descrizione_ente,provincia_sede,regione_sede,attivazioni,cessazioni,RATA,aggregation-time,qualifica,note,fonte_dato,Provincia Sede,CODICE ENTE,3descrizione,regione%sede,att ivazioni
0,6852caf7205349164bebcd69,11,2023,19,MINISTERO UNIVERSITA' E RICERCA,TO,01,250,40,202311,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,TO,19,MINISTERO UNIVERSITA' E RICERCA,1,250
1,6852caff6555da0907a40857,7,2023,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,BT,15,0,2,202307,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,BT,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,15,0
2,6852caef205349164bea2a6e,8,2021,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,Aq,12,0,4,202308,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,AQ,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,12,0
3,6852cb029952cb647ef389e4,4,2023,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,AQ,12,0,3,202304,2025-06-18T16:15:20.148346,Dirigente,NaN,NaN,AQ,2020,MINISTERO DELL'INTERNO - DIPARTIMENTO DELLA P.S.,12,0
4,6852cae9205349164be90859,6,2023,22,AGENZIA DELLE ENTRATE,VR,04,0,1,202306,2025-06-18T16:15:20.148346,Operatore,NaN,NaN,VR,22,AGENZIA DELLE ENTRATE,4,0


Below we inspect the data types and null counts for the second dataset to understand its starting state.

In [30]:
print("Shape:", df2.shape)
print("\nDtypes:")
print(df2.dtypes)
print("\nNull counts:")
print(df2.isnull().sum())

Shape: (20102, 19)

Dtypes:
_id                 object
mese                object
anno                object
codice_ente         object
descrizione_ente    object
provincia_sede      object
regione_sede        object
attivazioni         object
cessazioni          object
RATA                object
aggregation-time    object
qualifica           object
note                object
fonte_dato          object
Provincia Sede      object
CODICE ENTE          int64
3descrizione        object
regione%sede         int64
att ivazioni        object
dtype: object

Null counts:
_id                     0
mese                    0
anno                    0
codice_ente           210
descrizione_ente      445
provincia_sede        687
regione_sede          308
attivazioni             0
cessazioni              0
RATA                    0
aggregation-time        0
qualifica            5086
note                19802
fonte_dato          19942
Provincia Sede        367
CODICE ENTE             0
3descrizione   

### Step 1: Schema Validation

We repeat the same schema validation pipeline: compute pattern fingerprint, run type validation + correction, then naming check + correction.

In [31]:
pattern_report_2 = get_dataframe_patterns(df2)
pattern_report_2_str = json.dumps(pattern_report_2, indent=4)

print("DATA TYPE VALIDATION")
type_check_2 = schema_validator.run_validation_check(df2, DATA_PATH_2, pattern_report_2_str)
type_check_2_text = Outputs(type_check_2).get_text()
print(type_check_2_text)

DATA TYPE VALIDATION
Here's a list of columns with type mismatches I found:

- **mese**: Currently an object, but regex shows numeric patterns (N, NN); should be an int.
- **anno**: Currently an object, but regex shows numeric patterns (NNNN); should be an int.
- **codice_ente**: Currently an object, but regex shows numeric patterns (N, NN, NNN, NNNN); should be an int.
- **attivazioni**: Currently an object, but regex shows numeric patterns (N, NN, NNN, NNNN); should be a float (to account for comma/decimal separators).
- **cessazioni**: Currently an object, but regex shows numeric patterns (N, NN, NNN, NNNN); should be a float (to account for comma/decimal separators).
- **att ivazioni**: Currently an object, but regex shows numeric patterns (N, NN, NNN, NNNN); should be a float.


Below we inspect the data types and null counts for the second dataset to understand its starting state.

In [32]:
df2 = schema_validator.run_validation_correction(type_check_2_text, df2)
print("Type corrections applied. Updated dtypes:")
print(df2.dtypes)

Type corrections applied. Updated dtypes:
_id                  object
mese                  Int64
anno                  Int64
codice_ente           Int64
descrizione_ente     object
provincia_sede       object
regione_sede         object
attivazioni         float64
cessazioni          float64
RATA                 object
aggregation-time     object
qualifica            object
note                 object
fonte_dato           object
Provincia Sede       object
CODICE ENTE           int64
3descrizione         object
regione%sede          int64
att ivazioni        float64
dtype: object


Next, we run the naming convention check.

In [33]:
print("NAMING CONVENTION CHECK")
naming_check_2 = schema_validator.run_naming_check(df2, DATA_PATH_2)
naming_check_2_text = Outputs(naming_check_2).get_text()
print(naming_check_2_text)

NAMING CONVENTION CHECK
Here's a list of columns with naming violations I found:

- RATA: Currently uses uppercase, but should be lowercase to match the snake_case/lowercase convention of the other columns.
- aggregation-time: Uses a hyphen instead of an underscore, which is inconsistent with the snake_case style used elsewhere.
- Provincia Sede: Uses spaces and title casing, which deviates from the snake_case/lowercase standard.
- CODICE ENTE: Uses spaces and uppercase, which deviates from the snake_case/lowercase standard.
- 3descrizione: Starts with a numeric character, which is generally discouraged in naming conventions.
- regione%sede: Contains a special character (%), which is non-standard and can cause issues in programming environments.
- att ivazioni: Contains a space in the middle of the word, which is a naming error.


We apply the naming corrections automatically and display the updated column list.

In [34]:
df2 = schema_validator.run_naming_correction(naming_check_2_text, df2)
print("Naming corrections applied. Updated columns:")
print(list(df2.columns))

Naming corrections applied. Updated columns:
['_id', 'mese', 'anno', 'codice_ente', 'descrizione_ente', 'provincia_sede', 'regione_sede', 'attivazioni', 'cessazioni', 'rata', 'aggregation_time', 'qualifica', 'note', 'fonte_dato', 'provincia_sede_dup', 'codice_ente_dup', 'descrizione_3', 'regione_sede_dup', 'attivazioni_dup']


### Step 2: Completeness Analysis

We repeat the completeness pipeline: placeholder detection, per-column and per-row missing rates, and LLM summaries.

In [35]:
df2, placeholders_2 = completeness_analyst.run_completeness_analysis(df2, DATA_PATH_2)
print("Placeholders detected and replaced with NaN:", placeholders_2)

Placeholders detected and replaced with NaN: ['-', 'null', 'N/A', 'n/a', 'none', 'None', 'unknown', 'sconosciuto', 'N.D.', 'N/D', 'non disponibile', 'mancante', '/']


We compute the per-column missing-value percentages.

In [36]:
completeness_cols_2 = completeness_analyst.NA_percentages_columns(df2)
droppable_columns_2 = [col for col, v in completeness_cols_2.items() if v > 0.5]
overall_missing_2 = sum(completeness_cols_2.values()) / len(completeness_cols_2)

comp_df_2 = pd.DataFrame.from_dict(completeness_cols_2, orient="index", columns=["Missing %"])
comp_df_2 = comp_df_2.sort_values("Missing %", ascending=False)
comp_df_2["Missing %"] = comp_df_2["Missing %"].map(lambda x: f"{x:.2%}")
print(comp_df_2.to_string())
print(f"\nOverall missing: {overall_missing_2:.2%}")
print(f"Columns flagged for removal (>50% missing): {droppable_columns_2}")

                   Missing %
fonte_dato            99.20%
note                  98.51%
qualifica             25.30%
provincia_sede         3.42%
codice_ente            3.00%
descrizione_ente       2.21%
cessazioni             2.06%
attivazioni_dup        1.99%
attivazioni            1.99%
mese                   1.83%
provincia_sede_dup     1.83%
regione_sede           1.53%
codice_ente_dup        0.00%
regione_sede_dup       0.00%
descrizione_3          0.00%
_id                    0.00%
aggregation_time       0.00%
anno                   0.00%
rata                   0.00%

Overall missing: 12.78%
Columns flagged for removal (>50% missing): ['note', 'fonte_dato']


Next, we compute per-row completeness (excluding columns already flagged for removal).

In [37]:
completeness_rows_2 = completeness_analyst.NA_percentages_rows(df2, droppable_columns_2)
droppable_rows_2 = [key for key, v in completeness_rows_2.items() if v > 0.5]

print(f"Total rows: {len(completeness_rows_2)}")
print(f"Rows with >50% missing (excl. flagged columns): {len(droppable_rows_2)}")
print(f"Percentage: {len(droppable_rows_2)/len(completeness_rows_2):.2%}")

Total rows: 20102
Rows with >50% missing (excl. flagged columns): 0
Percentage: 0.00%


The LLM produces human-readable summaries of the column and row completeness findings.

In [38]:
summary_cols_2 = Outputs(
    completeness_analyst.summarize_columns(completeness_cols_2, overall_missing_2, droppable_columns_2, DATA_PATH_2)
).get_text()

summary_rows_2 = Outputs(
    completeness_analyst.summarize_rows(completeness_rows_2, droppable_rows_2, DATA_PATH_2)
).get_text()

completeness_report_2 = f"{summary_cols_2}\n\n{summary_rows_2}"
print(completeness_report_2)

### 1: Overall Completeness Summary
* **General Health:** The dataset is generally healthy and highly complete, with an overall missing value rate of approximately **12.78%**.
* **Data Integrity:** The majority of the core data fields are well-populated, suggesting that the dataset is reliable for most analytical purposes.
* **Distribution:** The missingness is not evenly distributed; while most columns are nearly complete, a small number of specific columns contain almost no data, which heavily influences the overall percentage.

### 2: Column Completeness Highlights

| Column Name | Missing Percentage |
| :--- | :--- |
| _id | 0.00% |
| anno | 0.00% |
| rata | 0.00% |
| aggregation_time | 0.00% |
| codice_ente_dup | 0.00% |
| regione_sede_dup | 0.00% |
| descrizione_3 | 0.00% |
| regione_sede | 1.53% |
| mese | 1.83% |
| provincia_sede_dup | 1.83% |
| attivazioni | 1.99% |
| attivazioni_dup | 1.99% |
| cessazioni | 2.06% |
| descrizione_ente | 2.21% |
| codice_ente | 3.00% |
| provin

### Step 3: Consistency Validation

We repeat the consistency pipeline: duplicate detection, format consistency, and cross-column logic.

In [39]:
duplicate_results_2 = consistency_validator.run_duplicate_detection(df2, DATA_PATH_2)

print(f"Exact row duplicates found (and removed): {duplicate_results_2['exact_duplicates']}")
print(f"Key column duplicates: {duplicate_results_2['key_column_duplicates']}")
print(f"Shape after deduplication: {df2.shape}")

Exact row duplicates found (and removed): 60
Key column duplicates: ["Column '_id' has 90 duplicate entries."]
Shape after deduplication: (20042, 19)


We recompute the pattern fingerprint after schema corrections and run the format consistency check.

In [40]:
pattern_report_2_updated = get_dataframe_patterns(df2)
pattern_report_2_updated_str = json.dumps(pattern_report_2_updated, indent=4)

format_results_2 = Outputs(
    consistency_validator.run_format_consistency_check(pattern_report_2_updated_str, df2, DATA_PATH_2)
).get_text()
print(format_results_2)

Based on the provided regex report and the rules for flagging, here are the columns with inconsistent patterns:

*   **mese**: The dominant pattern is `N.N` (14,696 rows), but there are 4,979 rows with `NN.N` and 367 `NULL` values.
*   **codice_ente**: The dominant pattern is `NN.N` (9,212 rows), but there are significant inconsistencies including `N.N` (6,496), `NNNN.N` (2,383), `NNN.N` (1,350), and `NULL` (601).
*   **attivazioni**: The dominant pattern is `N.N` (15,078 rows), with inconsistent patterns including `NN.N` (3,013), `NNN.N` (1,130), `NNNN.N` (399), `NULL` (399), `NNNNN.N` (16), `-N.N` (5), and `-NN.N` (2).
*   **cessazioni**: The dominant pattern is `N.N` (15,610 rows), with inconsistent patterns including `NN.N` (2,621), `NNN.N` (839), `NNNN.N` (535), `NULL` (414), `NNNNN.N` (16), `-N.N` (5), and `-NN.N` (2).
*   **rata**: The dominant pattern is `NNNNNN` (19,241 rows), with inconsistent patterns including `NNNN-NN` (290), `W-NNNN` (266), and `NN/NNNN` (245).
*   **aggr

We run the cross-column logic check on a clean sample (sparse columns and null rows removed).

In [41]:
df2_clean = df2.drop(columns=droppable_columns_2, errors='ignore').dropna(axis=0)
print(f"Clean sample shape: {df2_clean.shape}")

logic_results_2 = Outputs(
    consistency_validator.run_cross_column_logic(df2_clean)
).get_text()
print(logic_results_2)

Clean sample shape: (12738, 17)
As a Senior Data Quality Analyst, I have reviewed the provided sample. Below is the audit report regarding the logical consistency and structural integrity of the dataset.

### Inferred Logical Rules:
*   **Consistency of Entity Metadata:** The `codice_ente` and `descrizione_ente` must map consistently to the same entity across the entire dataset.
*   **Temporal Format Standardization:** The `aggregation_time` column should follow a single ISO 8601 format (e.g., `YYYY-MM-DDTHH:MM:SS.ssssss`).
*   **Rata Format Consistency:** The `rata` column (representing YYYYMM) should be a consistent integer or string format (e.g., `202407` vs `2024-10`).
*   **Data Integrity (Case Sensitivity):** Categorical fields like `provincia_sede` should be normalized (e.g., "fe" vs "FE").
*   **Logical Relationship:** `attivazioni` should match `attivazioni_dup`.

### Data Redundancy Alerts:
*   `codice_ente` and `codice_ente_dup` appear to contain identical information.
*   `

We assemble the full consistency report for the Remediator.

In [42]:
dupe_text_2 = (
    f"**Exact row duplicates found:** {duplicate_results_2['exact_duplicates']}\n\n"
    f"**Key column duplicates:** {duplicate_results_2['key_column_duplicates']}"
)
consistency_report_2 = "\n\n".join([
    "### Duplicate Analysis Results:",
    dupe_text_2.strip(),
    "### Format Consistency Analysis Results:",
    format_results_2.strip(),
    "### Cross Column Consistency Analysis Results:",
    logic_results_2.strip()
])
print("Consistency report assembled.")

Consistency report assembled.


### Step 4: Anomaly Detection

We repeat the anomaly detection pipeline: univariate outliers + categorical rare values.

In [43]:
print("UNIVARIATE OUTLIER DETECTION")
outlier_report_2, outlier_file_2 = anomaly_detector.univariate_outlier_detection(
    pattern_report_2_updated_str, df2, DATA_PATH_2
)
print(outlier_report_2)

UNIVARIATE OUTLIER DETECTION
Column 'attivazioni': Found 202 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Column 'cessazioni': Found 226 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Column 'attivazioni_dup': Found 200 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Remember that this analysis is done only on values that could be converted into a number. Refer to the report for the list of rows and values that cannot be cast into a number.
All the values that couldn't be cast have been ignored.


Now we run the categorical anomaly detection.

In [44]:
print("CATEGORICAL ANOMALY DETECTION")
cat_report_2, outlier_file_2 = anomaly_detector.categorical_outlier_detection(
    pattern_report_2_updated_str, df2, DATA_PATH_2, outlier_file_2
)
print(cat_report_2)

CATEGORICAL ANOMALY DETECTION
Column 'provincia_sede': Found 403 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.

Column 'qualifica': No outliers detected.

Column 'provincia_sede_dup': Found 80 outliers.
Details: Too many outliers to display (more than 10), check the final txt report for more details.




We assemble the full anomaly detection report for the Remediator.

In [45]:
anomaly_report_2 = "\n\n".join([
    "### Univariate Outlier Detection Results:",
    outlier_report_2.strip(),
    "### Categorical Outlier Detection Results:",
    cat_report_2.strip()
])
print("Anomaly report assembled.")

Anomaly report assembled.


### Step 5: Remediation Report

The Remediator aggregates all findings from Dataset 2 and produces the final Data Reliability Score and Action Plan.

In [46]:
remediation_2 = Outputs(
    remediator.generate_remediation_report(
        completeness_report_2, consistency_report_2, anomaly_report_2, DATA_PATH_2
    )
).get_text()
print(remediation_2)

## Final Data Reliability Score: 62/100
**Justification:** While core metrics are present, the dataset suffers from critical primary key duplicates (`_id`), systemic formatting inconsistencies in temporal fields (`rata`, `aggregation_time`), and significant data redundancy. The presence of logical contradictions between date-related columns makes the dataset currently unreliable for time-series analysis.

## Human-in-the-Loop Action Plan
Based on the reports, here are the exact steps the data engineering team should take to finish cleaning this dataset:

- **Missing Values:**
    - **Drop Columns:** Immediately remove `note` and `fonte_dato` as they provide no statistical value.
    - **Categorical Imputation:** For `qualifica` (25.3% missing), do not use mode imputation if the qualification is critical; instead, label missing values as `"Unknown"` to avoid biasing the analysis.
    - **Identifier Recovery:** For `codice_ente` (3% missing), attempt to recover values by cross-referencin

## 5: Conclusions

The AGENTS4DQ pipeline successfully processes both NoiPA datasets, demonstrating that a multi-agent architecture powered by LLMs can automate data quality assessment on heterogeneous administrative datasets without requiring hard-coded validation rules.

**Key findings across both experiments:**

- **Schema Validation**: the Schema Validator correctly identifies type mismatches (e.g., identifier columns stored as float/int instead of string, numeric columns stored as object) and naming violations (columns with special characters, leading digits, spaces, or inconsistent casing). Automatic corrections are applied successfully.

- **Completeness Analysis**: the Completeness Analyst detects placeholder values across both datasets and computes per-column and per-row missing rates. Columns like `note` and `fonte_dato` are flagged as largely empty and candidates for removal.

- **Consistency Validation**: the regex pattern fingerprinting approach proves effective at detecting format anomalies within columns. Cross-column logic checks identify redundant/mirror columns (e.g., `tipo_imposta` vs `Tipo Imposta`, `spesa` vs `SPESA TOTALE`, `cod_imposta` vs `2cod_imposta` vs `cod imposta ext`).

- **Anomaly Detection**: univariate outlier detection (3σ method) flags extreme values in numerical columns. Categorical anomaly detection identifies rare entity descriptions and tax types.

- **Remediation**: the Remediator produces actionable, human-readable action plans with specific imputation strategies and a global reliability score.

**The LLM-driven column selection makes the system dataset-agnostic**: no hard-coded rules need to change when switching from `spesa.csv` to `attivazioniCessazioni.csv`.

**Limitations and future work:**
- The current pipeline runs agents sequentially; parallel execution could improve latency.
- The remediation step produces suggestions rather than fully applying all fixes automatically, by design: high-confidence fixes (type casting, naming, deduplication) are applied, while domain-dependent decisions are left to the human operator.
- Expanding support beyond CSV to JSON and database connections would better match the full range of NoiPA data sources.
- Incorporating more sophisticated outlier detection methods (e.g., Isolation Forest, DBSCAN) alongside the current 3σ approach could improve detection accuracy on non-normal distributions.